# Kafka → Spark Streaming → Hive 실시간 파이프라인 실습

## spark - kafka 커넥터 확인

In [1]:
!docker exec -it spark-client sh -c 'ls /opt/bitnami/spark/jars/spark-sql-kafka-0-10_2.12-3.3.0.jar'
!docker exec -it spark-client sh -c 'ls /opt/bitnami/spark/jars/kafka-clients-3.3.1.jar'
!docker exec -it spark-client sh -c 'ls /opt/bitnami/spark/jars/spark-token-provider-kafka-0-10_2.12-3.3.0.jar'

!docker exec -it spark-master sh -c 'ls /opt/bitnami/spark/jars/spark-sql-kafka-0-10_2.12-3.3.0.jar'
!docker exec -it spark-master sh -c 'ls /opt/bitnami/spark/jars/kafka-clients-3.3.1.jar'
!docker exec -it spark-master sh -c 'ls /opt/bitnami/spark/jars/spark-token-provider-kafka-0-10_2.12-3.3.0.jar'

!docker exec -it spark-worker sh -c 'ls /opt/bitnami/spark/jars/spark-sql-kafka-0-10_2.12-3.3.0.jar'
!docker exec -it spark-worker sh -c 'ls /opt/bitnami/spark/jars/kafka-clients-3.3.1.jar'
!docker exec -it spark-worker sh -c 'ls /opt/bitnami/spark/jars/spark-token-provider-kafka-0-10_2.12-3.3.0.jar'

/opt/bitnami/spark/jars/spark-sql-kafka-0-10_2.12-3.3.0.jar
/opt/bitnami/spark/jars/kafka-clients-3.3.1.jar
/opt/bitnami/spark/jars/spark-token-provider-kafka-0-10_2.12-3.3.0.jar
/opt/bitnami/spark/jars/spark-sql-kafka-0-10_2.12-3.3.0.jar
/opt/bitnami/spark/jars/kafka-clients-3.3.1.jar
/opt/bitnami/spark/jars/spark-token-provider-kafka-0-10_2.12-3.3.0.jar
/opt/bitnami/spark/jars/spark-sql-kafka-0-10_2.12-3.3.0.jar
/opt/bitnami/spark/jars/kafka-clients-3.3.1.jar
/opt/bitnami/spark/jars/spark-token-provider-kafka-0-10_2.12-3.3.0.jar


In [3]:
! docker ps

CONTAINER ID   IMAGE                                     COMMAND                  CREATED             STATUS             PORTS                                                                    NAMES
06d9dc2dbab1   apache/airflow:2.9.0                      "/usr/bin/dumb-init …"   About an hour ago   Up About an hour   8080/tcp                                                                 airflow-scheduler
890425cb53d0   apache/airflow:2.9.0                      "/usr/bin/dumb-init …"   About an hour ago   Up About an hour   8080/tcp, 0.0.0.0:8099->8099/tcp                                         airflow-webserver
354628c5030c   postgres:13                               "docker-entrypoint.s…"   About an hour ago   Up About an hour   0.0.0.0:5433->5432/tcp                                                   airflow-postgresql
aac3c5304e6f   bde2020/hive:2.3.2-postgresql-metastore   "entrypoint.sh tail …"   About an hour ago   Up About an hour   10000/tcp, 10002/tcp                      

## SparkSession 구성 및 Kafka 연결

In [6]:
# 1. SparkSession 구성 및 Kafka 연결
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, DoubleType

spark = ( 
    SparkSession.builder 
    #.config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0")
    .appName("KafkaToHive") 
    .enableHiveSupport() 
    .getOrCreate()
)

In [7]:

raw_df = ( 
    spark.readStream 
    .format("kafka") 
    .option("kafka.bootstrap.servers", "kafka:9092") 
    .option("subscribe", "sales_topic") 
    .load()
)


In [8]:

schema = ( 
    StructType()
    .add("id", StringType())
    .add("category", StringType())
    .add("amount", DoubleType())
)

df = (
    raw_df.selectExpr("CAST(value AS STRING)") 
    .select(from_json(col("value"), schema).alias("data")) 
    .select("data.*")
)

In [9]:
# 2. Hive 테이블로 저장 (Parquet)
(
df.writeStream 
    .outputMode("append") 
    .format("parquet") 
    .option("path", "/user/hive/warehouse/sales_kafka_tbl/") 
    .option("checkpointLocation", "hdfs://hadoop-master:9000/checkpoints/sales_kafka_tbl/") 
    .start() 
    .awaitTermination()
)    

25/06/30 03:01:25 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
25/06/30 03:01:27 WARN NetworkClient: [Consumer clientId=consumer-spark-kafka-source-2b220732-63f3-4be8-8215-51c716341f2f-2007404865-driver-0-1, groupId=spark-kafka-source-2b220732-63f3-4be8-8215-51c716341f2f-2007404865-driver-0] Error while fetching metadata with correlation id 2 : {sales_topic=LEADER_NOT_AVAILABLE}
25/06/30 03:01:27 WARN NetworkClient: [Consumer clientId=consumer-spark-kafka-source-2b220732-63f3-4be8-8215-51c716341f2f-2007404865-driver-0-1, groupId=spark-kafka-source-2b220732-63f3-4be8-8215-51c716341f2f-2007404865-driver-0] Error while fetching metadata with correlation id 4 : {sales_topic=LEADER_NOT_AVAILABLE}


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/bitnami/python/lib/python3.8/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/bitnami/python/lib/python3.8/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/bitnami/python/lib/python3.8/socket.py", line 669, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 